# 데이터 라벨링

In [ ]:
# `preprocessing/`에 있는 로직을 사용하며, 이 노트북은 실행·확인·저장만 담당

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 설치 + MediaPipe 모델 다운로드
!pip install -q mediapipe

!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/1/face_landmarker.task
!wget -q -O pose_landmarker.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/1/pose_landmarker_full.task

In [ ]:
# repo 경로를 import 경로에 추가 (clone 위치에 맞게 수정)
import sys
REPO_DIR = "/content/drive/MyDrive/ai_study_flow_tracker"
if REPO_DIR not in sys.path:
    sys.path.append(REPO_DIR)

import pandas as pd
from preprocessing.pipeline import run

In [ ]:
video_path = "/content/drive/MyDrive/ai_study_flow_tracker/video_namwoo.mp4"

df, baseline, info = run(video_path, debug=False)  # EAR 디버그 출력이 필요하면 debug=True

t = info["timing"]
print(f"[전처리] 프레임 추출+mediapipe: {t['extract']:.1f}초 (총 {info['num_frames']}프레임)")
print(f"  face {t['face']:.2f}s / pose {t['pose']:.2f}s / skip-grab {t['skip']:.2f}s")
print(f"[점수 계산] {t['scoring']:.1f}초")
print(f"총 {info['num_segments']}개 세그먼트 계산 완료")
print(f"총 전처리 파이프라인: {t['total']:.1f}초")

In [ ]:
print("=== 개인화 Baseline ===")
for key, val in baseline.items():
    print(f"{key:20s}: {val:.4f}")

print(f"\n안정 구간 개수: {baseline['num_stable_segments']} / {min(120, info['num_segments'])}")

In [ ]:
print("=== 첫 10개 세그먼트 점수 비교 ===")
print(df[['segment_idx', 'final', 'personalized', 'personal_stability', 'sleepy_flag']].head(10).to_string(index=False))

print("\n=== 첫 10개 세그먼트 데이터 ===")
print(df.head(10).to_string(index=False))

print("\n=== 전체 통계 ===")
print(df[['final', 'personalized', 'personal_stability']].describe().round(3))

In [ ]:
# 피처별 고유값 개수 (상수 피처 진단용)
df.drop(columns=['segment_idx']).nunique()

In [ ]:
# 결과 저장
output_path = "/content/drive/MyDrive/ai_study_flow_tracker/labels_personalized_video_30.csv"
df.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"결과 저장 완료: {output_path}")